# Phase 0: Setup and test-set lock

This notebook loads the raw CDC BRFSS 2015 file, validates its schema, counts duplicate rows, and (after the duplicate-handling decision) creates the one-time stratified 80/20 split.

All logic lives in `src/features.py` and `src/evaluate.py`; this notebook only orchestrates.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src.features import (
    cast_to_schema, duplicate_report, file_sha256, largest_duplicate_groups,
    read_raw_csv, validate_schema,
)

config.seed_everything()
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 160)
print(f'SEED = {config.SEED}')
print(f'numpy {np.__version__}, pandas {pd.__version__}')

## 1. Load and validate the raw file

In [ ]:
raw = read_raw_csv()
raw_sha256 = file_sha256(config.RAW_FILE)
schema = validate_schema(raw)

print(f'File: {config.RAW_FILE.name}')
print(f'SHA-256: {raw_sha256}')
print(f'Shape: {raw.shape}  (expected {config.EXPECTED_ROWS:,} x {len(config.RAW_COLUMNS)})')
print(f'Row count matches expected: {schema.row_count_matches_expected}')
print(f'Columns match: {schema.columns_match}, order matches: {schema.column_order_matches}')
print(f'Errors: {schema.errors or "none"}')

schema_table = schema.summary_table()
schema_table.to_csv(config.RESULTS_DIR / 'phase0_schema_validation.csv')
schema_table

In [ ]:
assert schema.is_valid, schema.errors
df = cast_to_schema(raw)
print(df.dtypes.value_counts())
print(f'Positive rate (Diabetes_binary = 1): {df[config.TARGET].mean():.4%}')

## 2. Duplicate rows

In [ ]:
dups = duplicate_report(df)
dup_table = dups.summary_table()
dup_table.to_csv(config.RESULTS_DIR / 'phase0_duplicate_report.csv')
dup_table

In [ ]:
top_groups = largest_duplicate_groups(df, n=10)
top_groups.to_csv(config.RESULTS_DIR / 'phase0_largest_duplicate_groups.csv', index=False)
top_groups